In [6]:
!pip install open_clip_torch scikit-learn matplotlib umap-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 7.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
import os

# 1. Mount Google Drive into Colab
drive.mount('/content/drive')

# 2. Navigate to your project folder on Drive
# (Change 'ML_Assignment' to the exact name of the folder you created inside Drive)
%cd "/content/drive/MyDrive/ATML/PA1"
# Verify that Colab sees your local files
!ls task1

Mounted at /content/drive
/content/drive/MyDrive/ATML/PA1
analysis  make_cue_conflicts.py  models   scripts
data	  make_subset.py	 results  transforms.py


In [3]:
import os

# Create project directory structure
directories = [
    "task1/data",
    "task1/models",
    "task1/analysis",
    "task1/scripts",
    "task1/results/weights",
    "task1/results/plots",
    "task1/results/cue_conflicts"
]

for d in directories:
    os.makedirs(d, exist_ok=True)

# 1. task1/make_subset.py
with open("task1/make_subset.py", "w") as f:
    f.write('''"""
make_subset.py
"""
import os
import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset
import torchvision
from sklearn.model_selection import StratifiedShuffleSplit

SEED = 6304

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def prepare_stl10_splits(data_dir="./task1/data", seed=SEED):
    set_seed(seed)
    os.makedirs(data_dir, exist_ok=True)

    raw_train = torchvision.datasets.STL10(root=data_dir, split='train', download=True)
    raw_test = torchvision.datasets.STL10(root=data_dir, split='test', download=True)

    classes = raw_train.classes
    targets_train = np.array(raw_train.labels)
    targets_test = np.array(raw_test.labels)

    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, val_idx = next(sss.split(np.zeros(len(targets_train)), targets_train))

    num_classes = len(classes)
    samples_per_class = 500 // num_classes

    test_subset_indices = []
    for c in range(num_classes):
        class_indices = np.where(targets_test == c)[0]
        np.random.shuffle(class_indices)
        test_subset_indices.extend(class_indices[:samples_per_class].tolist())

    test_subset_indices = sorted(test_subset_indices)

    split_info = {
        "dataset": "STL-10",
        "classes": classes,
        "seed": seed,
        "train_indices": train_idx.tolist(),
        "val_indices": val_idx.tolist(),
        "test_subset_indices": test_subset_indices,
        "num_classes": num_classes
    }

    os.makedirs("./task1/results", exist_ok=True)
    with open("./task1/results/dataset_splits.json", "w") as f_out:
        json.dump(split_info, f_out, indent=2)

    print(f"[Dataset] STL-10 splits ready. Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_subset_indices)}")
    return split_info

class BaseDatasetWrapper(Dataset):
    def __init__(self, base_dataset, transform=None):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]
        img, target = item[0], item[1]
        if self.transform is not None:
            img = self.transform(img)
        return img, target, idx
''')

# 2. task1/models/backbones.py
with open("task1/models/backbones.py", "w") as f:
    f.write('''"""
models/backbones.py
"""
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import ResNet50_Weights, ViT_B_16_Weights
import open_clip

class FrozenBackbone(nn.Module):
    def __init__(self, model_type='resnet50', device='cuda'):
        super().__init__()
        self.model_type = model_type
        self.device = device

        if model_type == 'resnet50':
            weights = ResNet50_Weights.IMAGENET1K_V2
            backbone = models.resnet50(weights=weights)
            self.backbone = nn.Sequential(*list(backbone.children())[:-1])
            self.embed_dim = 2048
            self.mean = [0.485, 0.456, 0.406]
            self.std = [0.229, 0.224, 0.225]

        elif model_type == 'vit_b_16':
            weights = ViT_B_16_Weights.IMAGENET1K_V1
            self.backbone = models.vit_b_16(weights=weights)
            self.backbone.heads = nn.Identity()
            self.embed_dim = 768
            self.mean = [0.485, 0.456, 0.406]
            self.std = [0.229, 0.224, 0.225]

        elif model_type == 'clip_vit_b32':
            clip_model, _, _ = open_clip.create_model_and_transforms('ViT-B-32', pretrained='openai')
            self.backbone = clip_model.visual
            self.clip_model = clip_model
            self.embed_dim = 512
            self.mean = [0.48145466, 0.4578275, 0.40821073]
            self.std = [0.26862954, 0.26130258, 0.27577711]
            self.tokenizer = open_clip.get_tokenizer('ViT-B-32')

        for param in self.parameters():
            param.requires_grad = False
        self.eval()

    def forward(self, x):
    # Automatically resize 96x96 STL-10 images to 224x224 for ViT & CLIP
        if x.shape[-2:] != (224, 224):
            x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)

        with torch.no_grad():
            if self.model_type == 'resnet50':
                out = torch.flatten(self.backbone(x), 1)
            elif self.model_type == 'vit_b_16':
                out = self.backbone(x)
            elif self.model_type == 'clip_vit_b32':
                out = F.normalize(self.backbone(x), dim=-1)
        return out


class ClassifierHead(nn.Module):
    def __init__(self, embed_dim, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        return self.fc(x)


def train_classifier_head(backbone, train_loader, val_loader, num_classes=10, epochs=50, lr=1e-3, weight_decay=1e-4, patience=5, device='cuda', save_path=None):
    backbone.eval()
    head = ClassifierHead(backbone.embed_dim, num_classes).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    patience_counter = 0
    best_weights = None

    for epoch in range(epochs):
        head.train()
        for imgs, labels, _ in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            with torch.no_grad():
                feats = backbone(imgs)

            optimizer.zero_grad()
            logits = head(feats)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        head.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for imgs, labels, _ in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                feats = backbone(imgs)
                logits = head(feats)
                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += len(labels)

        val_acc = val_correct / val_total

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = head.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    if best_weights is not None:
        head.load_state_dict(best_weights)
        if save_path:
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            torch.save(best_weights, save_path)

    return head, best_val_acc


def zero_shot_clip_predict(clip_backbone, imgs, class_names, device='cuda'):
    clip_model = clip_backbone.clip_model.to(device)
    clip_model.eval()

    prompts = [f"a photo of a {c}" for c in class_names]
    tokens = clip_backbone.tokenizer(prompts).to(device)

    with torch.no_grad():
        text_features = F.normalize(clip_model.encode_text(tokens), dim=-1)
        image_features = clip_backbone(imgs.to(device))
        logit_scale = clip_model.logit_scale.exp()
        logits = logit_scale * (image_features @ text_features.T)
        probs = F.softmax(logits, dim=-1)

    return logits, probs
''')

# 3. task1/transforms.py
with open("task1/transforms.py", "w") as f:
    f.write('''"""
transforms.py
"""
import random
import numpy as np
import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from PIL import Image

class ImageNormalizer:
    def __init__(self, mean, std):
        self.norm = T.Normalize(mean=mean, std=std)

    def __call__(self, tensor):
        return self.norm(tensor)

def apply_grayscale(pil_img):
    return pil_img.convert('L').convert('RGB')

def apply_hue_rotation(pil_img, hue_factor=0.5):
    return TF.adjust_hue(pil_img, hue_factor)

def apply_translation(pil_img, dx, dy):
    w, h = pil_img.size
    pad_x, pad_y = abs(dx), abs(dy)
    padded = TF.pad(pil_img, padding=(pad_x, pad_y), padding_mode='reflect')
    return TF.crop(padded, top=pad_y - dy, left=pad_x - dx, height=h, width=w)

def get_patch_shuffled_image(pil_img, grid_size=4, seed=6304):
    w, h = pil_img.size
    patch_w, patch_h = w // grid_size, h // grid_size

    patches = [
        pil_img.crop((j * patch_w, i * patch_h, (j + 1) * patch_w, (i + 1) * patch_h))
        for i in range(grid_size) for j in range(grid_size)
    ]

    num_patches = grid_size * grid_size
    rng = np.random.RandomState(seed)

    perm = list(range(num_patches))
    while True:
        rng.shuffle(perm)
        if any(perm[i] != i for i in range(num_patches)):
            break

    shuffled_img = Image.new('RGB', (w, h))
    for idx, orig_idx in enumerate(perm):
        i, j = idx // grid_size, idx % grid_size
        shuffled_img.paste(patches[orig_idx], (j * patch_w, i * patch_h))

    return shuffled_img
''')

# 4. task1/make_cue_conflicts.py
with open("task1/make_cue_conflicts.py", "w") as f:
    f.write('''"""
make_cue_conflicts.py
"""
import os
import json
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.models as models
import torchvision.transforms.functional as TF
from PIL import Image
import numpy as np

CLASS_PAIRS = [('car', 'cat'), ('bird', 'truck'), ('dog', 'ship'), ('airplane', 'frog'), ('horse', 'deer')]

class VGGEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1).features
        self.slice = vgg[:21]
        for p in self.parameters():
            p.requires_grad = False

    def forward(self, x):
        return self.slice(x)

def visual_rejection_rule(stylized_pil, content_pil):
    stylized_gray = np.array(stylized_pil.convert('L'), dtype=np.float32) / 255.0
    content_gray = np.array(content_pil.convert('L'), dtype=np.float32) / 255.0

    if np.var(stylized_gray) < 0.01:
        return False

    gy_s, gx_s = np.gradient(stylized_gray)
    gy_c, gx_c = np.gradient(content_gray)
    corr = np.corrcoef(np.sqrt(gx_s**2 + gy_s**2).flatten(), np.sqrt(gx_c**2 + gy_c**2).flatten())[0, 1]

    return not (np.isnan(corr) or corr < 0.20)

def generate_cue_conflicts(dataset_wrapper, test_subset_indices, output_dir="./task1/results/cue_conflicts"):
    os.makedirs(output_dir, exist_ok=True)

    # 1. Recursively unwrap dataset_wrapper to find the raw root dataset
    root_ds = dataset_wrapper
    while hasattr(root_ds, 'base_dataset'):
        root_ds = root_ds.base_dataset
    while hasattr(root_ds, 'dataset'):
        root_ds = root_ds.dataset

    # 2. Get class names safely
    if hasattr(root_ds, 'classes'):
        classes = root_ds.classes
    else:
        classes = ['airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

    to_tensor, to_pil = T.ToTensor(), T.ToPILImage()

    # Helper function to extract samples directly from the root dataset
    def get_raw_sample(idx):
        item = root_ds[idx]
        img = item[0]
        target = item[1]

        if isinstance(target, torch.Tensor):
            target = int(target.item())
        else:
            target = int(target)

        if isinstance(img, torch.Tensor):
            img_tensor = img.cpu() if img.dim() == 3 else img.squeeze(0).cpu()
            img_pil = to_pil(img_tensor)
        else:
            img_pil = img

        return img_pil, target

    # 3. Build mapping from class name to indices using raw dataset
    class_to_indices = {}
    for idx in test_subset_indices:
        _, target = get_raw_sample(idx)
        class_name = classes[target]
        class_to_indices.setdefault(class_name, []).append(idx)

    accepted_count, rejected_count = 0, 0
    metadata = []

    # 4. Generate cue conflict image pairs
    for (cls_a, cls_b) in CLASS_PAIRS:
        for c_class, s_class in [(cls_a, cls_b), (cls_b, cls_a)]:
            c_indices, s_indices = class_to_indices.get(c_class, []), class_to_indices.get(s_class, [])

            for i in range(min(20, len(c_indices), len(s_indices))):
                c_idx, s_idx = c_indices[i], s_indices[i]
                c_img_pil, _ = get_raw_sample(c_idx)
                s_img_pil, _ = get_raw_sample(s_idx)

                c_tensor = to_tensor(c_img_pil).unsqueeze(0)
                s_tensor = to_tensor(s_img_pil).unsqueeze(0)

                stylized_tensor = torch.clamp(TF.gaussian_blur(c_tensor, kernel_size=[5, 5]) * 0.4 + s_tensor * 0.6, 0.0, 1.0)
                stylized_pil = to_pil(stylized_tensor.squeeze(0))

                if visual_rejection_rule(stylized_pil, c_img_pil):
                    filename = f"conflict_{c_class}_style_{s_class}_{i}.png"
                    filepath = os.path.join(output_dir, filename)
                    stylized_pil.save(filepath)
                    metadata.append({
                        "filepath": filepath, "content_class": c_class, "style_class": s_class
                    })
                    accepted_count += 1
                else:
                    rejected_count += 1

    # 5. Save metadata JSON output
    metadata_path = "./task1/results/cue_conflicts_metadata.json"
    os.makedirs(os.path.dirname(metadata_path), exist_ok=True)
    with open(metadata_path, "w") as f:
        json.dump({"accepted_count": accepted_count, "rejected_count": rejected_count, "conflicts": metadata}, f, indent=2)

    return metadata
''')

# 5. task1/analysis/evaluate_bias.py
with open("task1/analysis/evaluate_bias.py", "w") as f:
    f.write('''"""
analysis/evaluate_bias.py
"""
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(y_true, logits_or_probs):
    # Ensure y_true is a numpy array
    if isinstance(y_true, torch.Tensor):
        y_true = y_true.detach().cpu().numpy()

    # Process logits / probabilities
    if isinstance(logits_or_probs, torch.Tensor):
        probs = F.softmax(logits_or_probs, dim=-1).detach().cpu().numpy()
    else:
        probs = np.array(logits_or_probs)

    preds = np.argmax(probs, axis=1)

    return {
        "accuracy": float(accuracy_score(y_true, preds)),
        "macro_f1": float(f1_score(y_true, preds, average='macro')),
        "mean_max_confidence": float(np.mean(np.max(probs, axis=1))),
        "predictions": preds.tolist()
    }

def compute_prediction_consistency(clean_preds, transformed_preds):
    # Ensure inputs are converted properly if they are tensors
    if isinstance(clean_preds, torch.Tensor):
        clean_preds = clean_preds.detach().cpu().numpy()
    if isinstance(transformed_preds, torch.Tensor):
        transformed_preds = transformed_preds.detach().cpu().numpy()

    clean_arr = np.array(clean_preds)
    trans_arr = np.array(transformed_preds)

    return float(np.mean(clean_arr == trans_arr))

def evaluate_cue_conflicts(model_predict_fn, conflict_metadata, classes):
    class_to_idx = {c: i for i, c in enumerate(classes)}
    n_shape, n_texture, n_other = 0, 0, 0

    for item in conflict_metadata:
        shape_label = class_to_idx[item["content_class"]]
        texture_label = class_to_idx[item["style_class"]]
        pred_class, _ = model_predict_fn(item["filepath"])

        # Ensure pred_class is a standard Python int
        if isinstance(pred_class, torch.Tensor):
            pred_class = int(pred_class.detach().cpu().item())
        elif isinstance(pred_class, str) and pred_class in class_to_idx:
            pred_class = class_to_idx[pred_class]
        else:
            pred_class = int(pred_class)

        if pred_class == shape_label:
            n_shape += 1
        elif pred_class == texture_label:
            n_texture += 1
        else:
            n_other += 1

    n_total = len(conflict_metadata)
    denom = n_shape + n_texture
    return {
        "N_shape": n_shape, "N_texture": n_texture, "N_other": n_other, "N_total": n_total,
        "shape_bias_percent": float((n_shape / denom * 100.0) if denom > 0 else 0.0),
        "coverage_percent": float(((n_shape + n_texture) / n_total * 100.0) if n_total > 0 else 0.0)
    }
''')

# 6. task1/analysis/feature_similarity.py
with open("task1/analysis/feature_similarity.py", "w") as f:
    f.write('''"""
analysis/feature_similarity.py
"""
import torch
import torch.nn.functional as F

def compute_cosine_stability(clean_features, transformed_features):
    clean_norm = F.normalize(torch.tensor(clean_features, dtype=torch.float32), p=2, dim=-1)
    trans_norm = F.normalize(torch.tensor(transformed_features, dtype=torch.float32), p=2, dim=-1)
    cosine_sims = torch.sum(clean_norm * trans_norm, dim=-1)

    return {
        "cosine_stability_mean": float(torch.mean(cosine_sims).item()),
        "cosine_stability_std": float(torch.std(cosine_sims).item())
    }
''')

# 7. task1/analysis/representation.py
with open("task1/analysis/representation.py", "w") as f:
    f.write('''"""
analysis/representation.py
"""
import os
import matplotlib.pyplot as plt
import numpy as np
from sklearn.manifold import TSNE

SEED = 6304

def visualize_combined_representation(clean_feats, transformed_feats, labels, class_names, backbone_name, intervention_name, save_dir="./task1/results/plots"):
    os.makedirs(save_dir, exist_ok=True)
    N = len(clean_feats)
    combined = np.concatenate([clean_feats, transformed_feats], axis=0)

    embeds_2d = TSNE(n_components=2, random_state=SEED, perplexity=30, learning_rate='auto', init='pca').fit_transform(combined)
    clean_2d, trans_2d = embeds_2d[:N], embeds_2d[N:]

    plt.figure(figsize=(9, 7))
    cmap = plt.cm.get_cmap("tab10", len(class_names))

    for c_idx in range(len(class_names)):
        mask = (labels == c_idx)
        plt.scatter(clean_2d[mask, 0], clean_2d[mask, 1], c=[cmap(c_idx)], marker='o', alpha=0.6, edgecolors='k')
        plt.scatter(trans_2d[mask, 0], trans_2d[mask, 1], c=[cmap(c_idx)], marker='^', alpha=0.6, edgecolors='r')

    plt.title(f"{backbone_name.upper()}: Clean (O) vs {intervention_name.title()} (^)")
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"{backbone_name}_{intervention_name}_tsne.png"), dpi=200)
    plt.close()
''')

# 8. task1/scripts/run_task1.py
with open("task1/scripts/run_task1.py", "w") as f:
    f.write('''"""
scripts/run_task1.py
"""
import os
import sys
import json
import torch
import torch.nn as nn
import numpy as np
from PIL import Image
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as T

sys.path.append("./task1")

from make_subset import prepare_stl10_splits, BaseDatasetWrapper, SEED, set_seed
from models.backbones import FrozenBackbone, train_classifier_head, zero_shot_clip_predict
try:
    from models.backbones import ClassifierHead
except ImportError:
    ClassifierHead = lambda in_dim, out_dim: nn.Linear(in_dim, out_dim)

from transforms import ImageNormalizer, apply_grayscale, apply_hue_rotation, apply_translation, get_patch_shuffled_image
from make_cue_conflicts import generate_cue_conflicts
from analysis.evaluate_bias import compute_metrics, compute_prediction_consistency, evaluate_cue_conflicts
from analysis.feature_similarity import compute_cosine_stability
from analysis.representation import visualize_combined_representation

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def main():
    set_seed(SEED)
    print(f"Executing Task 1 Pipeline on device: {DEVICE}")

    # Ensure required results directories exist
    ckpt_dir = "./task1/results/checkpoints"
    plots_dir = "./task1/results/plots"
    os.makedirs(ckpt_dir, exist_ok=True)
    os.makedirs(plots_dir, exist_ok=True)

    split_info = prepare_stl10_splits(data_dir="./task1/data", seed=SEED)
    classes = split_info["classes"]

    raw_train = torchvision.datasets.STL10(root="./task1/data", split='train', download=False)
    raw_test = torchvision.datasets.STL10(root="./task1/data", split='test', download=False)

    train_subset = Subset(raw_train, split_info["train_indices"])
    val_subset = Subset(raw_train, split_info["val_indices"])
    test_eval_subset = Subset(raw_test, split_info["test_subset_indices"])

    test_wrapper = BaseDatasetWrapper(test_eval_subset)
    test_labels = np.array([test_wrapper[i][1] for i in range(len(test_wrapper))])

    backbones = {
        'resnet50': FrozenBackbone('resnet50', device=DEVICE).to(DEVICE),
        'vit_b_16': FrozenBackbone('vit_b_16', device=DEVICE).to(DEVICE),
        'clip_vit_b32': FrozenBackbone('clip_vit_b32', device=DEVICE).to(DEVICE)
    }

    heads = {}
    to_tensor = T.ToTensor()

    print("\n[1/6] Training or Loading Classifier Heads...")
    for model_name, backbone in backbones.items():
        ckpt_path = os.path.join(ckpt_dir, f"{model_name}_head.pt")

        if os.path.exists(ckpt_path):
            print(f"  -> Found saved checkpoint for {model_name}. Loading weights...")
            head = ClassifierHead(backbone.embed_dim, len(classes)).to(DEVICE)
            head.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
            head.eval()
            heads[model_name] = head
        else:
            print(f"  -> No checkpoint found for {model_name}. Training classifier head...")
            norm = ImageNormalizer(backbone.mean, backbone.std)
            transform_fn = lambda img: norm(to_tensor(img))

            train_ds = BaseDatasetWrapper(train_subset, transform=transform_fn)
            val_ds = BaseDatasetWrapper(val_subset, transform=transform_fn)

            head, best_val_acc = train_classifier_head(
                backbone, DataLoader(train_ds, batch_size=64, shuffle=True),
                DataLoader(val_ds, batch_size=64, shuffle=False),
                num_classes=len(classes), epochs=50, lr=1e-3, weight_decay=1e-4, patience=5, device=DEVICE
            )
            torch.save(head.state_dict(), ckpt_path)
            heads[model_name] = head
            print(f"  -> {model_name} Val Acc: {best_val_acc:.4f} (Saved checkpoint to {ckpt_path})")

    clean_test_tensors = {
        m: torch.stack([ImageNormalizer(backbones[m].mean, backbones[m].std)(to_tensor(test_wrapper[i][0])) for i in range(len(test_wrapper))])
        for m in backbones
    }

    results = {"clean_baseline": {}}
    clean_preds_dict, clean_feats_dict = {}, {}

    print("\n[2/6] Evaluating Clean Baselines...")
    for model_name in backbones:
        imgs = clean_test_tensors[model_name].to(DEVICE)
        feats = backbones[model_name](imgs)
        logits = heads[model_name](feats)

        m = compute_metrics(test_labels, logits)
        results["clean_baseline"][model_name] = m
        clean_preds_dict[model_name] = m["predictions"]
        clean_feats_dict[model_name] = feats.cpu().numpy()
        print(f"  -> {model_name} Top-1 Acc: {m['accuracy']:.4f}")

    # Zero-Shot CLIP
    _, clip_probs = zero_shot_clip_predict(backbones['clip_vit_b32'], clean_test_tensors['clip_vit_b32'], classes, device=DEVICE)
    results["clean_baseline"]["clip_zero_shot"] = compute_metrics(test_labels, clip_probs)

    print("\n[3/6] Evaluating Color Interventions...")
    for name, tf_fn in [("grayscale", apply_grayscale), ("hue_rotation", apply_hue_rotation)]:
        for m_name in backbones:
            t_imgs = torch.stack([ImageNormalizer(backbones[m_name].mean, backbones[m_name].std)(to_tensor(tf_fn(test_wrapper[i][0]))) for i in range(len(test_wrapper))]).to(DEVICE)
            logits = heads[m_name](backbones[m_name](t_imgs))
            m = compute_metrics(test_labels, logits)
            m["consistency"] = compute_prediction_consistency(clean_preds_dict[m_name], m["predictions"])
            results.setdefault("color_bias", {}).setdefault(name, {})[m_name] = m

    print("\n[4/6] Evaluating Cue Conflicts...")
    conflict_metadata = generate_cue_conflicts(test_wrapper, split_info["test_subset_indices"])
    for m_name in backbones:
        norm = ImageNormalizer(backbones[m_name].mean, backbones[m_name].std)
        def predict_fn(path):
            img = norm(to_tensor(Image.open(path).convert('RGB'))).unsqueeze(0).to(DEVICE)
            logits = heads[m_name](backbones[m_name](img))
            return logits.argmax(dim=-1).item(), torch.softmax(logits, dim=-1).max().item()
        results.setdefault("cue_conflict", {})[m_name] = evaluate_cue_conflicts(predict_fn, conflict_metadata, classes)

    print("\n[5/6] Patch Shuffling & Representation Stability...")
    for m_name in backbones:
        norm = ImageNormalizer(backbones[m_name].mean, backbones[m_name].std)
        shuff_imgs = torch.stack([norm(to_tensor(get_patch_shuffled_image(test_wrapper[i][0], 4, SEED))) for i in range(len(test_wrapper))]).to(DEVICE)
        shuff_feats = backbones[m_name](shuff_imgs)

        m = compute_metrics(test_labels, heads[m_name](shuff_feats))
        m["consistency"] = compute_prediction_consistency(clean_preds_dict[m_name], m["predictions"])
        results.setdefault("patch_shuffle", {})[m_name] = m

        results.setdefault("representation_stability", {})[m_name] = compute_cosine_stability(clean_feats_dict[m_name], shuff_feats.cpu().numpy())
        visualize_combined_representation(clean_feats_dict[m_name], shuff_feats.cpu().numpy(), test_labels, classes, m_name, "patch_shuffle")

    with open("./task1/results/task1_final_results.json", "w") as f:
        json.dump(results, f, indent=2)

    print("\nSuccess! Results written to ./task1/results/task1_final_results.json")

if __name__ == "__main__":
    main()
''')

print("All repository files generated successfully under task1/")

All repository files generated successfully under task1/


In [7]:
!python task1/scripts/run_task1.py

Executing Task 1 Pipeline on device: cuda
100% 2.64G/2.64G [12:29<00:00, 3.52MB/s]  
[Dataset] STL-10 splits ready. Train: 4000, Val: 1000, Test: 500
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 211MB/s]
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100% 330M/330M [00:03<00:00, 115MB/s] 

open_clip_model.safetensors: downloading bytes:  58% 349M/605M [00:03<00:01, 210MB/s, 28.9MB/s  ]  
open_clip_model.safetensors: downloading bytes:  71% 432M/605M [00:03<00:00, 199MB/s, 36.3MB/s  ]
open_clip_model.safetensors: reconstructing file:  55% 335M/605M [00:04<00:03, 86.3MB/s, 25.0MB/s  ]
open_clip_model.safetensors: downloading bytes: 100% 432M/432M [00:05<00:00, 79.5MB/s, 37.6MB/s  ]] 
open_clip_model.safetensors: reconstructing file: 100% 605M/605M [00:05<00:00, 111MB/s, 52.2MB/s  ]
/usr